In [0]:
import json

spark.sql("USE CATALOG `mvp-base_programação_operacional`")

df = spark.sql("""
    SELECT
        f.status_unificado,
        f.valor_recorrente,
        t.territorio,
        tm.dia_semana
    FROM gold.fato_agendamento f
    LEFT JOIN gold.dim_territorio t ON f.sk_territorio = t.sk_territorio
    LEFT JOIN gold.dim_tempo tm ON f.sk_tempo = tm.sk_tempo
""").toPandas()

# Pergunta 1: oportunidades por território
p1 = (
    df[df["status_unificado"] == "oportunidade"]["territorio"]
    .value_counts()
    .sort_values(ascending=False)
)

# Pergunta 2: status por dia da semana
p2 = (
    df.dropna(subset=["dia_semana"])
    .groupby(["dia_semana", "status_unificado"])
    .size()
    .unstack(fill_value=0)
)

# Pergunta 3: receita por território, com % acumulado (Pareto)
p3 = (
    df.dropna(subset=["valor_recorrente"])
    .groupby("territorio")["valor_recorrente"]
    .sum()
    .sort_values(ascending=False)
)
p3_pct_acumulado = (p3.cumsum() / p3.sum() * 100).round(1)

# Monta o JSON no formato pronto para consumir num dashboard Chart.js
dashboard_json = {
    "pergunta_1_oportunidades_por_territorio": {
        "labels": p1.index.tolist(),
        "valores": p1.values.tolist(),
    },
    "pergunta_2_status_por_dia_semana": {
        "dias": p2.index.tolist(),
        "series": {col: p2[col].tolist() for col in p2.columns},
    },
    "pergunta_3_receita_por_territorio": {
        "labels": p3.index.tolist(),
        "receita": p3.values.tolist(),
        "pct_acumulado": p3_pct_acumulado.tolist(),
    },
}

print(json.dumps(dashboard_json, indent=2, ensure_ascii=False))

# Salva o JSON no mesmo Volume que já existe (bronze/raw_extraction),
# pronto pra ser consumido por um HTML/Chart.js
caminho_json = "/Volumes/mvp-base_programação_operacional/bronze/raw_extraction/dashboard_respostas.json"
with open(caminho_json, "w", encoding="utf-8") as f:
    json.dump(dashboard_json, f, ensure_ascii=False, indent=2)

print(f"\nJSON salvo em: {caminho_json}")

In [0]:
spark.sql("USE CATALOG `mvp-base_programação_operacional`")

### Pergunta 1: Qual território concentra mais oportunidades de agendamento não realizadas?

In [0]:
display(spark.sql("""
    SELECT t.territorio, COUNT(*) as oportunidades
    FROM gold.fato_agendamento f
    JOIN gold.dim_territorio t ON f.sk_territorio = t.sk_territorio
    WHERE f.status_unificado = 'oportunidade'
    GROUP BY t.territorio
    ORDER BY oportunidades DESC
"""))

### Pergunta 2: Existe padrão de dia da semana na taxa de agendamentos cumpridos vs. atrasados?

In [0]:
display(spark.sql("""
    SELECT tm.dia_semana, f.status_unificado, COUNT(*) as total
    FROM gold.fato_agendamento f
    JOIN gold.dim_tempo tm ON f.sk_tempo = tm.sk_tempo
    WHERE tm.dia_semana != 'Não aplicável'
    GROUP BY tm.dia_semana, f.status_unificado
"""))

### Pergunta 3: A receita prevista está concentrada em poucos territórios (Pareto 80/20)?

In [0]:
display(spark.sql("""
    SELECT t.territorio, SUM(f.valor_recorrente) as receita
    FROM gold.fato_agendamento f
    JOIN gold.dim_territorio t ON f.sk_territorio = t.sk_territorio
    WHERE f.valor_recorrente IS NOT NULL
    GROUP BY t.territorio
    ORDER BY receita DESC
"""))

### Extra: Simulação de uso dos dados com saída em painel para apresentação 

In [0]:
spark.sql("USE CATALOG `mvp-base_programação_operacional`")

import json

df = spark.sql("""
    SELECT
        f.status_unificado,
        f.valor_recorrente,
        t.territorio,
        tm.dia_semana
    FROM gold.fato_agendamento f
    LEFT JOIN gold.dim_territorio t ON f.sk_territorio = t.sk_territorio
    LEFT JOIN gold.dim_tempo tm ON f.sk_tempo = tm.sk_tempo
""").toPandas()

p1 = (
    df[df["status_unificado"] == "oportunidade"]["territorio"]
    .value_counts()
    .sort_values(ascending=False)
)

p2 = (
    df.dropna(subset=["dia_semana"])
    .groupby(["dia_semana", "status_unificado"])
    .size()
    .unstack(fill_value=0)
)

p3 = (
    df.dropna(subset=["valor_recorrente"])
    .groupby("territorio")["valor_recorrente"]
    .sum()
    .sort_values(ascending=False)
)
p3_pct_acumulado = (p3.cumsum() / p3.sum() * 100).round(1)

dashboard_json = {
    "pergunta_1_oportunidades_por_territorio": {
        "labels": p1.index.tolist(),
        "valores": p1.values.tolist(),
    },
    "pergunta_2_status_por_dia_semana": {
        "dias": p2.index.tolist(),
        "series": {col: p2[col].tolist() for col in p2.columns},
    },
    "pergunta_3_receita_por_territorio": {
        "labels": p3.index.tolist(),
        "receita": p3.values.tolist(),
        "pct_acumulado": p3_pct_acumulado.tolist(),
    },
}

HTML_TEMPLATE = '''
<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<title>Painel de Agendamentos B2B</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;600;700&family=IBM+Plex+Sans:wght@400;500;600&family=IBM+Plex+Mono:wght@400;500&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.0/chart.umd.min.js"></script>
<style>
  :root{
    --bg: #14171B;
    --panel: #1B2027;
    --panel-2: #20262E;
    --border: #2B323C;
    --text: #E7E9EC;
    --text-dim: #8D97A3;
    --amber: #E8A33D;
    --amber-dim: #6B5326;
    --teal: #4FBBAA;
    --teal-dim: #285048;
    --coral: #D9694F;
    --coral-dim: #5A3327;
  }
  *{box-sizing:border-box;}
  body{
    margin:0;
    background:var(--bg);
    color:var(--text);
    font-family:'IBM Plex Sans', sans-serif;
    min-height:100vh;
  }
  header{
    padding:2.5rem 3rem 1.5rem;
    border-bottom:1px solid var(--border);
  }
  header h1{
    font-family:'Space Grotesk', sans-serif;
    font-weight:600;
    font-size:1.9rem;
    margin:0 0 0.35rem;
    letter-spacing:-0.01em;
  }
  header p{
    margin:0;
    color:var(--text-dim);
    font-size:0.95rem;
    max-width:640px;
    line-height:1.5;
  }
  #meta{
    margin-top:1rem;
    display:flex;
    gap:2rem;
    font-family:'IBM Plex Mono', monospace;
    font-size:0.78rem;
    color:var(--text-dim);
  }
  #meta span b{color:var(--text);font-weight:500;}
  main{
    display:grid;
    padding:2rem 3rem 3rem;
    grid-template-columns:repeat(3, 1fr);
    gap:1.5rem;
  }
  section.panel{
    background:var(--panel);
    border:1px solid var(--border);
    border-radius:4px;
    padding:1.5rem;
    display:flex;
    flex-direction:column;
    gap:1rem;
  }
  section.panel h2{
    font-family:'Space Grotesk', sans-serif;
    font-size:1.02rem;
    font-weight:600;
    margin:0;
  }
  section.panel .insight{
    font-size:0.83rem;
    color:var(--text-dim);
    line-height:1.5;
    margin:0;
    padding-bottom:0.75rem;
    border-bottom:1px solid var(--border);
  }
  .chart-wrap{position:relative; height:220px;}
  table{
    width:100%;
    border-collapse:collapse;
    font-size:0.8rem;
    font-family:'IBM Plex Mono', monospace;
  }
  table th{
    text-align:left;
    color:var(--text-dim);
    font-weight:500;
    padding:0.4rem 0.3rem;
    border-bottom:1px solid var(--border);
    font-size:0.72rem;
  }
  table td{
    padding:0.4rem 0.3rem;
    border-bottom:1px solid #20252c;
  }
  table td.num{text-align:right;}
</style>
</head>
<body>

<header>
  <h1>Painel de agendamentos B2B</h1>
  <p>Leitura das 3 perguntas de negócio definidas no objetivo do MVP, geradas a partir de <code>gold.fato_agendamento</code>.</p>
  <div id="meta"></div>
</header>

<main id="dashboard">
  <section class="panel">
    <h2>Oportunidades por território</h2>
    <p class="insight" id="insight-1"></p>
    <div class="chart-wrap"><canvas id="chart1"></canvas></div>
    <table id="table1"></table>
  </section>

  <section class="panel">
    <h2>Status por dia da semana</h2>
    <p class="insight" id="insight-2"></p>
    <div class="chart-wrap"><canvas id="chart2"></canvas></div>
    <table id="table2"></table>
  </section>

  <section class="panel">
    <h2>Receita por território (Pareto)</h2>
    <p class="insight" id="insight-3"></p>
    <div class="chart-wrap"><canvas id="chart3"></canvas></div>
    <table id="table3"></table>
  </section>
</main>

<script>
const ORDEM_DIAS = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"];
const NOME_DIA = {Monday:"Seg", Tuesday:"Ter", Wednesday:"Qua", Thursday:"Qui", Friday:"Sex", Saturday:"Sáb", Sunday:"Dom"};

const DASHBOARD_DATA = __DASHBOARD_DATA__;

function renderDashboard(data){
  document.getElementById('meta').innerHTML =
    `<span>fonte <b>gold.fato_agendamento</b></span><span>gerado <b>${new Date().toLocaleDateString('pt-BR')}</b></span>`;
  renderPergunta1(data.pergunta_1_oportunidades_por_territorio);
  renderPergunta2(data.pergunta_2_status_por_dia_semana);
  renderPergunta3(data.pergunta_3_receita_por_territorio);
}

renderDashboard(DASHBOARD_DATA);

function fmtInt(n){ return new Intl.NumberFormat('pt-BR').format(Math.round(n)); }
function fmtBRL(n){ return new Intl.NumberFormat('pt-BR', {style:'currency', currency:'BRL', maximumFractionDigits:0}).format(n); }

function renderPergunta1(p1){
  const top = p1.labels[0];
  document.getElementById('insight-1').textContent =
    `${top} concentra o maior número de agendamentos sem data confirmada (${fmtInt(p1.valores[0])} de ${fmtInt(p1.valores.reduce((a,b)=>a+b,0))} oportunidades em aberto).`;

  new Chart(document.getElementById('chart1'), {
    type: 'bar',
    data: {
      labels: p1.labels,
      datasets: [{
        data: p1.valores,
        backgroundColor: p1.labels.map((_,i)=> i===0 ? '#E8A33D' : '#3A4048'),
        borderRadius: 2,
        barThickness: 22,
      }]
    },
    options: {
      indexAxis: 'y',
      plugins: {
        legend: { display:false },
        tooltip: { backgroundColor:'#20262E', borderColor:'#2B323C', borderWidth:1, titleFont:{family:'IBM Plex Sans'}, bodyFont:{family:'IBM Plex Mono'} },
      },
      scales: {
        x: { ticks:{color:'#8D97A3', font:{family:'IBM Plex Mono', size:10}}, grid:{color:'#242A32'} },
        y: { ticks:{color:'#E7E9EC', font:{family:'IBM Plex Sans', size:11}}, grid:{display:false} },
      }
    }
  });

  const t1 = document.getElementById('table1');
  t1.innerHTML = '<tr><th>Território</th><th style="text-align:right">Oportunidades</th></tr>' +
    p1.labels.map((l,i)=>`<tr><td>${l}</td><td class="num">${fmtInt(p1.valores[i])}</td></tr>`).join('');
}

function renderPergunta2(p2){
  const diasReais = ORDEM_DIAS.filter(d => p2.dias.includes(d));
  const idx = d => p2.dias.indexOf(d);
  const agendado = diasReais.map(d => p2.series.agendado[idx(d)] || 0);
  const atrasado = diasReais.map(d => p2.series.atrasado[idx(d)] || 0);
  const oportunidade = diasReais.map(d => p2.series.oportunidade ? (p2.series.oportunidade[idx(d)] || 0) : 0);

  const totalAgendado = agendado.reduce((a,b)=>a+b,0);
  const totalAtrasado = atrasado.reduce((a,b)=>a+b,0);
  const piorDiaIdx = atrasado.indexOf(Math.max(...atrasado));
  document.getElementById('insight-2').textContent =
    `${NOME_DIA[diasReais[piorDiaIdx]]} concentra o maior volume de atrasos (${fmtInt(atrasado[piorDiaIdx])}). No total, agendado supera atrasado em ${fmtInt(totalAgendado - totalAtrasado)} contratos.`;

  new Chart(document.getElementById('chart2'), {
    type: 'bar',
    data: {
      labels: diasReais.map(d => NOME_DIA[d]),
      datasets: [
        { label:'Agendado', data: agendado, backgroundColor:'#4FBBAA', borderRadius:2, maxBarThickness:36 },
        { label:'Atrasado', data: atrasado, backgroundColor:'#D9694F', borderRadius:2, maxBarThickness:36 },
        { label:'Oportunidade', data: oportunidade, backgroundColor:'#8B7FD8', borderRadius:2, maxBarThickness:36 },
      ]
    },
    options: {
      plugins: {
        legend: { position:'bottom', labels:{color:'#8D97A3', font:{family:'IBM Plex Sans', size:10}, boxWidth:8, padding:16} },
        tooltip: { backgroundColor:'#20262E', borderColor:'#2B323C', borderWidth:1, titleFont:{family:'IBM Plex Sans'}, bodyFont:{family:'IBM Plex Mono'} },
      },
      scales: {
        x: { stacked:true, ticks:{color:'#E7E9EC', font:{family:'IBM Plex Sans', size:11}}, grid:{display:false} },
        y: { stacked:true, ticks:{color:'#8D97A3', font:{family:'IBM Plex Mono', size:10}}, grid:{color:'#242A32'} },
      }
    }
  });

  const t2 = document.getElementById('table2');
  t2.innerHTML = '<tr><th>Dia</th><th style="text-align:right">Agendado</th><th style="text-align:right">Atrasado</th><th style="text-align:right">Oportunidade</th></tr>' +
    diasReais.map((d,i)=>`<tr><td>${NOME_DIA[d]}</td><td class="num">${fmtInt(agendado[i])}</td><td class="num">${fmtInt(atrasado[i])}</td><td class="num">${fmtInt(oportunidade[i])}</td></tr>`).join('');
}

function renderPergunta3(p3){
  const idx80 = p3.pct_acumulado.findIndex(v => v >= 80);
  document.getElementById('insight-3').textContent = idx80 >= 0
    ? `Os ${idx80+1} primeiros territórios concentram ${p3.pct_acumulado[idx80]}% da receita prevista — padrão Pareto confirmado.`
    : `Receita distribuída de forma mais uniforme entre territórios, sem concentração clara de Pareto.`;

  new Chart(document.getElementById('chart3'), {
    data: {
      labels: p3.labels,
      datasets: [
        {
          type:'bar',
          label:'Receita (R$)',
          data: p3.receita,
          backgroundColor:'#E8A33D',
          borderRadius:2,
          order:2,
          yAxisID:'y',
        },
        {
          type:'line',
          label:'% acumulado',
          data: p3.pct_acumulado,
          borderColor:'#4FBBAA',
          backgroundColor:'#4FBBAA',
          tension:0.25,
          pointRadius:3,
          order:1,
          yAxisID:'y1',
        }
      ]
    },
    options: {
      plugins: {
        legend: { position:'bottom', labels:{color:'#8D97A3', font:{family:'IBM Plex Sans', size:10}, boxWidth:8, padding:16} },
        tooltip: { backgroundColor:'#20262E', borderColor:'#2B323C', borderWidth:1, titleFont:{family:'IBM Plex Sans'}, bodyFont:{family:'IBM Plex Mono'} },
      },
      scales: {
        x: { ticks:{color:'#E7E9EC', font:{family:'IBM Plex Sans', size:10}}, grid:{display:false} },
        y: { position:'left', ticks:{color:'#8D97A3', font:{family:'IBM Plex Mono', size:9}, callback:v=>fmtBRL(v)}, grid:{color:'#242A32'} },
        y1: { position:'right', min:0, max:100, ticks:{color:'#4FBBAA', font:{family:'IBM Plex Mono', size:9}, callback:v=>v+'%'}, grid:{display:false} },
      }
    }
  });

  const t3 = document.getElementById('table3');
  t3.innerHTML = '<tr><th>Território</th><th style="text-align:right">Receita</th><th style="text-align:right">% acum.</th></tr>' +
    p3.labels.map((l,i)=>`<tr><td>${l}</td><td class="num">${fmtBRL(p3.receita[i])}</td><td class="num">${p3.pct_acumulado[i]}%</td></tr>`).join('');
}
</script>

</body>
</html>
'''

html_final = HTML_TEMPLATE.replace("__DASHBOARD_DATA__", json.dumps(dashboard_json, ensure_ascii=False))
displayHTML(html_final)